# Replication Methodology Details

## Data-Cleaning Pipeline {#sec-data-cleaning}

This appendix gives the full step-by-step replication pipeline summarised in Section 2.1.

We begin with the CRSP monthly returns file: CRSP marks each fund's first-return observation with an "R" code, which we replace by missing, and we drop rows with no `crsp_fundno`. From the CRSP fund summary we drop exchange-traded products (non-empty `et_flag`), construct a binary index-fund flag from `index_fund_flag` values D, B, and E, and build a tri-state equity indicator from the `crsp_obj_cd` prefix (ED or EF for equity, any other non-empty prefix for non-equity, missing otherwise). We retain CRSP's composition shares (`per_com`, `per_pref`, `per_cash`, and the other `per_*` columns) for the @berk-2015 footnote-9 holdings screen used later.

We merge Morningstar Direct on standardised ticker × month and then re-attempt unmatched rows on unambiguous CUSIP-8 × month links. We cross-validate the two return series and apply two safeguards: ticker matches are rejected when CRSP-only evidence (objective code or footnote-9 holdings) contradicts a Morningstar equity classification, or when more than half of at least twelve overlapping months disagree by over 500 basis points. In addition, a deterministic return-repair pass switches CRSP to Morningstar only when a NAV-implied return (from `mnav`) is materially closer to Morningstar than to CRSP. Remaining missing CRSP returns are then filled from Morningstar. Missing total net assets are filled from Morningstar's share-class field and, when that is also missing, from the replicated fund-level aggregate. Observations from the aggregate are tagged so the later portfolio-level TNA roll-up does not double-count them. All Morningstar fields are converted to CRSP units (millions of dollars; decimal fees) at import.

We then classify each fund-month as actively managed, ordinary index, or leveraged/inverse index. A Morningstar category containing "Index" is the primary signal, then CRSP's `index_fund_flag`. For the remainder we apply the BvB Table 1 keyword set (INDEX, S&P, RUSSELL, SPDR, ETF, ISHARES, and similar) to the concatenated CRSP and Morningstar fund names, refuted by Table 2 (SELECT, MANAGED, ENHANCED, PLUS, target-date years between 1970 and 2045). Within the index-flagged set, Table 3 keywords (INVERSE, ULTRA, nX patterns) identify leverage, refuted by Table 4 (SHORT TERM, LONG/SHORT). We record a status switch when `index_fund_flag` changes mid-life and both the pre- and post-change periods exceed twelve months, following the @berk-2015 conversion rule.

The equity filter is applied in two passes. The first is a time-aware majority rule. A fund is equity when more than 50% of its observations with any classification signal so indicate, drawing on `crsp_obj_cd`, Morningstar Category Group, Morningstar US Category Group, and `ms_equity_pct`. The footnote-9 override drops any fund whose CRSP average holdings show under 50% in stocks or over 20% in cash. The second pass is a portfolio-level classifier run after share-class aggregation, with priority (i) the CRSP holdings screen (common, preferred, and other-equity shares against cash) when composition data exist, (ii) the Morningstar `ms_equity_pct` portfolio mean when composition is missing, (iii) a metadata fallback ordered MS Category Group, `crsp_obj_cd`, Lipper class, and (iv) a fund-name keyword pass for bond, money-market, commodity, balanced, target-date, and allocation products. Portfolios labelled non-equity, allocation, or unclassified are dropped.

Share classes are grouped into portfolios using a CRSP-first connected-components procedure. Each share class is parsed into a CRSP main name and a subclass suffix by splitting `fund_name` on the rightmost semicolon or slash; the main name, any unambiguous ticker, `crsp_portno`, and `ms_fundid` each provide an edge, with ambiguous tickers and Morningstar identifiers. Those linked to more than one CRSP main name across the panel are excluded. A minimum-group-id propagation is iterated over the four edge types to convergence. Within a portfolio-month, net returns and the expense ratio are TNA-weighted across share classes. Portfolio TNA is the maximum of the genuine share-class TNA sum and the maximum replicated Morningstar fund-level aggregate, which avoids double-counting on multi-class funds for which Morningstar's fund-level total was copied to every share class. A portfolio is labelled as index whenever any of its share classes is so flagged.

Fees are imputed in stages: negatives are set to missing; CRSP gaps are filled from Morningstar after dividing by one hundred to convert percentage points to decimal fractions. We then fill within-fund means inside the fund fiscal year inferred from `fiscal_yearend`, falling back to the calendar year when fiscal-year information is unavailable. Management fees alone are additionally filled across share classes within the same portfolio-fiscal-year, while expense ratios are not cross-filled because they legitimately differ across classes. Expense ratios above 5% per year and management fees above 3% per year are capped to missing, and funds with no usable fee data across all observations are dropped. We merge monthly CPI (CPIAUCSL) from FRED, rebase it to January 2000 = 1, and compute inflation-adjusted TNA as `rsize` = `mtna` / `cpiid`. The remaining gaps are supplemented from Morningstar and then by carrying the most recent past `mtna` forward within each share class, and rows with no usable AUM are dropped. The @berk-2015 \$5 million absorbing-state filter and 24-month tenure requirement are portfolio-level screens applied after share-class aggregation: for each portfolio we identify the first month in which `port_rsize` reaches \$5 million in January 2000 dollars and retain every subsequent observation, including months in which inflation-adjusted size temporarily falls back below the threshold; portfolios that never cross the threshold or with fewer than 24 post-screen months are dropped.

Gross portfolio returns are computed as $R^g_{i,t} = R^n_{i,t} + f_{i,t}$, where $f_{i,t}$ is the contemporaneous TNA-weighted annual expense ratio for month $t$ divided by twelve, matching equation (3) of @berk-2015. We do not lag the fee. For the Vanguard benchmark we download the eleven Vanguard index funds listed in the @berk-2015 data appendix and, for each, keep Investor-class returns before the Admiral-class inception date and Admiral-class returns thereafter, since Admiral shares carry lower fees. CRSP's Small-Cap Index Fund (NAESX) includes returns from an actively managed predecessor before October 1989. These pre-index returns are set to missing. Pre-inception months are handled by the augmented/zeros OLS construction documented in Section 2.1. For the FFC benchmark we use the market excess return, SMB, HML, and momentum factors that mirror Kenneth French's data library. As a final step we drop any share-class month with an absolute return exceeding 100% before aggregation, treating these as data errors.